# Portfolio Risk Engine — Production Demonstration

Institutional-grade quantitative risk analysis covering:
1. Data ingestion and validation
2. Portfolio construction with Euler risk allocation
3. Historical, Parametric, GARCH-enhanced, and Monte Carlo (Cholesky) VaR
4. GARCH(1,1) volatility modelling with diagnostics
5. Kupiec POF, Christoffersen independence, and Basel backtesting
6. Risk dashboard

**Author**: Quantitative Risk Engineering
**Version**: 1.0.0

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from risk_engine import (
    DataFetcher, Portfolio,
    HistoricalVaR, ParametricVaR, MonteCarloVaR, VaRResult,
    GARCHModel, GARCHParams,
    VaRBacktest, BacktestMetrics,
    RiskVisualizer
)

# Run parameters
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'NVDA', 'META']
LOOKBACK = 7560
NOTIONAL = 1_000_000
CL = 0.95
MC_SIMS = 100_000
print('Portfolio Risk Engine v1.0.0 — initialized')

## 1. Market Data

Fetch adjusted close prices with automatic validation, imputation, and stale-price detection.

In [ ]:
fetcher = DataFetcher(tickers=TICKERS, lookback_days=LOOKBACK)
prices = fetcher.fetch_data()
returns = fetcher.get_returns()

print(f'Period:  {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Obs:     {len(prices):,}')
print(f'Assets:  {list(prices.columns)}')

fetcher.get_stats()[0]  # Preview first asset stats

## 2. Portfolio Construction

Equal-weight portfolio with Euler risk contribution decomposition.

In [ ]:
portfolio = Portfolio(returns=returns, name='Tech-Finance Portfolio')
print(portfolio.summary())

### Risk Contribution (Euler Allocation)

$\text{CTE}_i = w_i \cdot \frac{(\Sigma w)_i}{\sigma_p}$

In [ ]:
rc = portfolio.get_risk_contribution()
rc.style.format('{:.2%}', subset=['weight', 'pct_contrib']).format('{:.6f}', subset=['marginal_risk', 'risk_contrib'])

## 3. Value at Risk — All Models

Compute VaR via four methodologies.  All results use the `VaRResult` dataclass contract.

In [ ]:
pr = portfolio.get_portfolio_returns()
w_arr = portfolio.get_weights().values

# 3.1 Historical VaR (non-parametric, overlapping periods for hp>1)
h = HistoricalVaR(pr, confidence_level=CL, notional=NOTIONAL, bootstrap=True, random_state=42).calculate()

# 3.2 Parametric VaR (normal, standard volatility)
p = ParametricVaR(pr, confidence_level=CL, notional=NOTIONAL, volatility_model='standard').calculate()

# 3.3 Parametric VaR (EWMA, lambda=0.94)
e = ParametricVaR(pr, confidence_level=CL, notional=NOTIONAL, volatility_model='ewma').calculate()

# 3.4 Monte Carlo VaR (Cholesky, correlated simulation)
m = MonteCarloVaR(
    pr, confidence_level=CL, notional=NOTIONAL,
    n_simulations=MC_SIMS, random_state=42,
    use_cholesky=True, component_returns=returns, component_weights=w_arr
).calculate()

# 3.5 GARCH-enhanced Parametric VaR
garch = GARCHModel(pr, distribution='normal').fit()
g_vol = garch.forecast_vol()
g = ParametricVaR(pr, confidence_level=CL, notional=NOTIONAL, daily_vol=g_vol).calculate()

comparison = pd.DataFrame({
    'Model':        ['Historical', 'Parametric', 'Parametric-EWMA', 'MonteCarlo-Cholesky', 'GARCH-Parametric'],
    'VaR ($)':      [f'${x.var_abs:,.0f}'   for x in [h, p, e, m, g]],
    'ES ($)':       [f'${x.es_abs:,.0f}'    for x in [h, p, e, m, g]],
    'VaR (bps)':    [f'{x.var_pct*10000:.1f}' for x in [h, p, e, m, g]],
    'ES (bps)':     [f'{x.es_pct*10000:.1f}'  for x in [h, p, e, m, g]],
})
comparison.style.hide(axis='index')

### GARCH(1,1) Diagnostics

In [ ]:
print(garch.summary())

## 4. Backtesting

Rolling Historical VaR → Kupiec POF, Christoffersen independence, Basel traffic light.

In [ ]:
window = 252
var_roll = pd.Series(index=pr.index[window:], dtype=float)
for i in range(window, len(pr)):
    var_roll.iloc[i-window] = HistoricalVaR(
        pr.iloc[i-window:i], confidence_level=CL
    ).calculate().var_pct

test_r = pr[var_roll.index]
bt = VaRBacktest(test_r, var_roll, confidence_level=CL)
metrics = bt.run()

print(f'Observations:     {metrics.n_obs:,}')
print(f'Violations:       {metrics.n_violations} ({metrics.violation_rate:.2%})')
print(f'Expected rate:    {metrics.expected_rate:.2%}')
print(f'\nKupiec POF        LR={metrics.kupiec_lr:.3f}  p={metrics.kupiec_pvalue:.4f}  [{"PASS" if metrics.kupiec_pass else "FAIL"}]')
print(f'Independence      LR={metrics.ind_lr:.3f}  p={metrics.ind_pvalue:.4f}  [{"PASS" if metrics.ind_pass else "FAIL"}]')
print(f'Conditional Cov   LR={metrics.cc_lr:.3f}  p={metrics.cc_pvalue:.4f}  [{"PASS" if metrics.cc_pass else "FAIL"}]')
print(f'Basel Zone:       {metrics.basel_zone.upper()}')

## 5. Visualisation

In [ ]:
viz = RiskVisualizer()

fig = viz.dashboard(
    portfolio=portfolio,
    var_results={'Historical': h.var_pct, 'Parametric': p.var_pct, 'MonteCarlo': m.var_pct},
    cond_vol=garch.conditional_vol(),
    drawdown=portfolio.get_drawdown_series()
)
plt.show()

## 6. Tail Risk Analysis

In [ ]:
s = portfolio.get_statistics()
print(f'5% Tail Ratio:    {s["tail_ratio_5"]:.2f}  (upside/downside tail ratio)')
print(f'1% Tail Ratio:    {s["tail_ratio_1"]:.2f}')
print(f'Skewness:         {s["skewness"]:.2f}')
print(f'Excess Kurtosis:  {s["kurtosis"]:.2f}')

print('\nWorst 10 Days:')
for d, v in pr.nsmallest(10).items():
    print(f'  {d.strftime("%Y-%m-%d")}  {v:>8.4%}')

---

## Appendix: API Reference

### VaRResult (immutable dataclass)
- `var_pct` — VaR as positive decimal (e.g. 0.02 = 2% loss)
- `es_pct` — Expected Shortfall as positive decimal
- `var_abs` / `es_abs` — In currency units
- `confidence` / `holding_period` / `notional` / `model_name`

### GARCHParams (immutable dataclass)
- `omega`, `alpha`, `beta`, `persistence`, `half_life`
- `unconditional_variance`, `unconditional_volatility`

### BacktestMetrics (immutable dataclass)
- `n_obs`, `n_violations`, `violation_rate`, `expected_rate`
- `kupiec_lr`, `kupiec_pvalue`, `kupiec_pass`
- `ind_lr`, `ind_pvalue`, `ind_transition`, `ind_pass`
- `cc_lr`, `cc_pvalue`, `cc_pass`
- `basel_zone`